In [ ]:
import os
import requests
import json

from pathlib import Path
from datasets import load_dataset

import sys
sys.path.append("..")

In [ ]:
data_path = Path("../data")
ckpt_path = Path("../ckpt")

ckpt_path.mkdir(exist_ok=True, parents=True)

warandpeace_path = data_path / "warandpeace.txt"


tokenizer_path = ckpt_path / 'tokenizer'
tokenizer_path.mkdir(exist_ok=True, parents=True)

In [ ]:
with open(warandpeace_path, 'r') as file:
    warandpeace = file.read()

In [ ]:
chars = sorted(list(set(warandpeace)))
vocab_size = len(chars)

print("".join(chars))
print(vocab_size)

In [ ]:
text = warandpeace[:100]

tokens = text.encode("utf-8")
tokens = list(map(int, tokens))
tokens

print("text", len(text))
print("tokens", len(tokens))

In [ ]:
ord("п")

In [ ]:
from collections import defaultdict

In [ ]:

def get_stats(ids: list[int]):
    counts = defaultdict(int)
    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts

stats = get_stats(tokens)
print(stats)

In [ ]:
sorted(stats, reverse=True, key=lambda x: stats.get(x))


chr(ord("ф"))

In [ ]:
bytes([208, 190]).decode("utf-8")

In [ ]:
def merge(ids, pair, idx):
    newids = []

    i = 0

    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [ ]:
top_pair = max(stats, key=stats.get)

tokens = merge(tokens, top_pair, 256)

In [ ]:
print(len(tokens))

In [ ]:

from tqdm import tqdm

In [ ]:
###

text = warandpeace
tokens = text.encode("utf-8")
tokens = list(map(int, tokens))

vocab_size = 1000
num_merges = vocab_size - 256
ids = list(tokens)

merges = {}
with tqdm(range(num_merges), desc="Merging") as pbar:
    for i in range(num_merges):
        stats = get_stats(ids)
        pair = max(stats, key=stats.get)
        idx = 256 + i
        pbar.set_postfix({
            "merge": f"{pair}",
            "idx": f"{idx}",
            "compression_rate": f"{len(tokens) / len(ids):.3f}"
        })
        ids = merge(ids, pair, idx)
        merges[pair] = idx

In [ ]:
print(len(tokens))
print(len(ids))

In [ ]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text


In [ ]:
vocab = {idx: bytes([idx]) for idx in range(256)}
vocab

In [ ]:
def encode(text):
    tokens = list(text.encode("utf-8"))

    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens



In [ ]:
text =  "привет, мир и война будет ли работать с 🤪 эмодзи? или english? --, /"
encoded = encode(text)
decoded = decode(encoded)
print(encoded)
print(decoded)
print(len(text), len(encoded))

In [ ]:
list("🤪".encode("utf-8"))

In [ ]:
import regex as re

gpt2pattern = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

In [ ]:
print(re.findall(gpt2pattern, text))